# Lab 1 · Notebook 02 — NumPy and Pandas

**Course:** Artificial Intelligence with Python  
**Estimated duration:** 55 minutes  
**Prerequisites:** Notebook 01 (Python fundamentals), local Jupyter environment

### Learning outcomes
By the end of this notebook, you should be able to:

- Create NumPy arrays and use vectorized operations.
- Load a CSV dataset using Pandas.
- Inspect a DataFrame, select columns and rows, and filter observations.
- Compute descriptive statistics and summaries for groups.
- Recognize missing values and distinguish numerical from categorical columns.

**How to work:** Run code cells from top to bottom. Complete cells marked **Exercise**; they intentionally do not contain solutions. Use **Shift+Enter** to execute a cell.

**Dataset:** `data/apartments.csv` contains **150 fictional rental listings**, generated for this lab. We will reuse it for exploratory analysis, preprocessing and regression later. All rents are illustrative, not market estimates.

## 1. Import libraries (2 min)

Python libraries are imported with `import`. In NumPy and Pandas, aliases `np` and `pd` are conventional.

In [ ]:
import numpy as np
import pandas as pd

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 2. NumPy: arrays instead of loops (12 min)

A NumPy array is a collection of values that supports efficient *element-wise* operations. Java programmers can think of it as a typed array with rich numerical operations.

Unlike a regular Python list, adding a number to a NumPy array changes each entry.

In [ ]:
measurements = np.array([18.2, 20.0, 21.5, 19.8, 23.1])
print("Readings:", measurements)
print("Shape:", measurements.shape)
print("Type:", measurements.dtype)
print("Readings + 1.5:", measurements + 1.5)
print("Mean:", measurements.mean())
print("Above 20 °C:", measurements[measurements > 20])

### Python list versus NumPy array

In [ ]:
values = [2, 4, 6]
print("Python list * 2:", values * 2)
print("NumPy array * 2:", np.array(values) * 2)

matrix = np.array([[1, 2, 3], [4, 5, 6]])
print("Matrix shape:", matrix.shape)
print("Second column:", matrix[:, 1])

### Exercise 1 — Calculate with an array (4 min)

Given five weekly electricity consumption measurements in kWh:

1. Convert the Python list into a NumPy array.
2. Find the **mean** and **maximum** consumption.
3. Create an array of measurements that are **greater than 15 kWh**.
4. Convert the measurements into Wh by multiplying by 1000.

**Self-check:** The mean is `15.4` kWh, and two values are greater than 15.

In [ ]:
consumption_kwh = [12, 15, 18, 11, 21]

# TODO: convert to a NumPy array
# TODO: print mean and maximum
# TODO: filter values greater than 15
# TODO: convert to Wh

## 3. Reading a CSV file with Pandas (10 min)

Pandas represents tables as **DataFrames** (rows = observations, columns = variables). We will use a CSV of fictional apartment listings.

**Important:** Launch JupyterLab from the repository root and open this notebook inside `lab01/`. When a notebook runs, relative paths typically resolve from the notebook's directory. The file should be located at `lab01/data/apartments.csv`.

In [ ]:
from pathlib import Path

csv_path = Path("data/apartments.csv")
if not csv_path.exists():
    raise FileNotFoundError(
        f"Cannot find {csv_path.resolve()}. "
        "Check that this notebook is in lab01/ and apartments.csv is in lab01/data/."
    )

df = pd.read_csv(csv_path)
print("Rows, columns:", df.shape)
display(df.head())

### Understanding the columns

| Column | Meaning | Kind |
|---|---|---|
| `apartment_id` | Synthetic listing identifier | identifier |
| `area_m2` | Floor area in square metres | numerical |
| `rooms` | Number of rooms | numerical (discrete) |
| `building_age_years` | Building age | numerical, some missing |
| `distance_to_center_km` | Distance to city centre | numerical, some missing |
| `district` | Fictional city district | categorical |
| `balcony` | Whether a balcony is present | categorical |
| `energy_rating` | Illustrative letter rating | categorical, some missing |
| `monthly_rent_eur` | Monthly rent in euros | **potential regression target** |

The missing cells are intentional: they will become important in the next laboratory.

In [ ]:
print("Column names:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nNumerical summary:")
display(df.describe())

### Exercise 2 — Inspect the dataset (4 min)

1. Show the **last 5 rows**.
2. Print the number of observations (rows).
3. Find the **minimum, maximum and average monthly rent**.
4. Show the unique district names.

Use `df.tail()`, `len(df)`, column methods such as `.min()`, and `.unique()`.

In [ ]:
# TODO: show the last five rows
# TODO: print the number of rows
# TODO: print min, max and mean rent
# TODO: print unique districts

## 4. Selecting and filtering data (12 min)

You can select one column as a **Series** (`df["area_m2"]`) or multiple columns as a DataFrame (`df[[...]]`).

- `.iloc[...]` selects by integer position.
- Boolean masks select rows satisfying conditions.

As in NumPy, boolean masks help you avoid writing loops over the rows.

In [ ]:
display(df[["district", "area_m2", "monthly_rent_eur"]].head())
display(df.iloc[:3, :4])

large_apartments = df[df["area_m2"] >= 100]
print("At least 100 m²:", len(large_apartments))
display(large_apartments.head())

central_with_balcony = df[(df["district"] == "Central") & (df["balcony"] == "yes")]
print("Central with balcony:", len(central_with_balcony))

### Exercise 3 — Filter listings (5 min)

1. Find listings with **at least 3 rooms** and rent **below €1,500**.
2. Keep only `apartment_id`, `rooms`, `district`, and `monthly_rent_eur`.
3. Sort the results by rent in ascending order, and display the first five.

**Hint:** Combine conditions with `&` and wrap each condition in parentheses. Use `.sort_values(...)`.

In [ ]:
# TODO: combine the two conditions
# TODO: select the four requested columns
# TODO: sort by monthly_rent_eur and display the first five

## 5. Grouping, missing values, and new columns (12 min)

`groupby` summarizes observations within categories. Missing values are often represented as `NaN` (not a number). They should be identified before using the data for machine learning.

**Do not fill in missing values yet** — that is a main topic of Lab 2.

In [ ]:
print("Average rent by district (€):")
display(df.groupby("district")["monthly_rent_eur"].mean().round(0).sort_values())

print("\nMissing values per column:")
print(df.isna().sum())

# DataFrame.assign returns a new DataFrame; it does not overwrite df.
df_with_rate = df.assign(rent_per_m2=df["monthly_rent_eur"] / df["area_m2"])
display(df_with_rate[["apartment_id", "rent_per_m2"]].head())

### Exercise 4 — Summarize and interpret (5 min)

1. Calculate the **median** monthly rent per district.
2. Count listings by `balcony` value.
3. Identify which columns contain missing data.
4. Create a `rent_per_m2` column, then find its average (do not overwrite the original `df` unless you want to).

Write one sentence: **Why is average rent alone potentially misleading when comparing apartments of different sizes?**

In [ ]:
# TODO: median rent by district
# TODO: count listings with and without balcony
# TODO: list columns containing at least one missing value
# TODO: average rent per m²

# Interpretation (write your answer in a Markdown cell or comment):

## 6. Mini-challenge — Make a data-backed observation (5 min)

Choose one claim to investigate:

- **A.** Apartments closer to the city centre tend to have higher rents.
- **B.** Apartments with balconies tend to have higher rents.
- **C.** Larger apartments tend to have higher monthly rents.

Use Pandas filtering, grouping or a correlation coefficient to gather evidence. In **2–3 sentences**, summarize your observation and mention **one possible confounding factor**. A pattern in this fictional dataset does not establish causation.

In [ ]:
# TODO: explore one hypothesis with Pandas
# Suggested starting points:
# df[["distance_to_center_km", "monthly_rent_eur"]].corr()
# df.groupby("balcony")["monthly_rent_eur"].mean()
# df[["area_m2", "monthly_rent_eur"]].corr()

## Wrap-up and next steps

You have worked with the basic tools of tabular data analysis:

**NumPy arrays → Pandas DataFrames → CSV import → selection/filtering → summaries → missing-data inspection.**

In **Notebook 03**, you will use visualizations to explore distributions and relationships. In **Lab 2**, you will decide how to handle missing values and prepare features for model training.

### Exit questions

1. How is a NumPy array different from a Python list?
2. What is the difference between a Pandas Series and DataFrame?
3. Why might a column of identifiers be unhelpful as a machine-learning feature?
4. Which column in this dataset could be a regression target?